# 02. GNSS-SDR 수신기 처리와 Doppler 검증

## 연구 목적
정상 IQ를 GNSS-SDR에 입력한 뒤 acquisition·tracking·관측값을 확인하고 PRN별 Doppler를 시각화합니다.

## 입력
- 01단계의 정상 IQ와 manifest
- `artifacts/receiver_runs/*/observables.csv`

현재 GNSS-SDR 자동 실행 모듈이 연결되기 전에는 준비 상태와 기대 schema를 점검합니다.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("저장소 루트 또는 notebooks/에서 Notebook을 실행하세요.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
ARTIFACTS = PROJECT_ROOT / "artifacts"
print("PROJECT_ROOT:", PROJECT_ROOT)


## 중간 확인 1 — 01단계 입력과 02단계 준비 상태

In [ ]:
from gnss_doppler_lab.research_sequence import latest_run, sequence_status

normal_run = latest_run(ARTIFACTS / 'rf_runs')
status = sequence_status(ARTIFACTS)['02_receiver_processing']
print('정상 IQ:', normal_run / 'gps_l1ca_s8_iq.bin')
print('GNSS-SDR observables 준비:', status['ready'])
print('경로:', status['path'] or '아직 없음 — GNSS-SDR 연동 단계 필요')

## 중간 확인 2 — 관측값 표와 PRN별 Doppler
CSV가 있으면 자동 표시하고, 없으면 필요한 열을 안내합니다.

In [ ]:
import csv
import matplotlib.pyplot as plt

required = {'time', 'prn', 'doppler_hz'}
obs_path = Path(status['path']) / 'observables.csv' if status['ready'] else None
rows = list(csv.DictReader(obs_path.open())) if obs_path else []
if not rows:
    print('관측값 없음. 기대 열:', sorted(required))
else:
    missing = required - rows[0].keys()
    if missing:
        raise ValueError(f'observables.csv 누락 열: {sorted(missing)}')
    print('행 수:', len(rows), 'PRN:', sorted({r['prn'] for r in rows}))
    fig, ax = plt.subplots(figsize=(13, 5))
    for prn in sorted({r['prn'] for r in rows}):
        group = [r for r in rows if r['prn'] == prn]
        ax.plot([float(r['time']) for r in group], [float(r['doppler_hz']) for r in group], label=prn)
    ax.set(title='GNSS-SDR measured Doppler', xlabel='Time (s)', ylabel='Doppler (Hz)')
    ax.grid(alpha=.25); ax.legend(ncol=4); plt.show()

## 판정

- [ ] 예상 가시 PRN이 acquisition된다.
- [ ] acquisition Doppler가 search 범위 안에 있다.
- [ ] tracking lock과 C/N₀가 시간에 따라 유지된다.
- [ ] PRN별 측정 Doppler와 simulator/orbit truth의 부호·크기가 일치한다.
- [ ] observables와 실행 설정이 run 단위로 보존된다.

## 다음 단계
정상 수신기 체인이 검증되면 `03_normal_vs_spoofing_comparison.ipynb`에서 공격 전후를 동일 기준으로 비교합니다.